In [1]:
print("Hello, World!")

Hello, World!


## Cell 1 — Setup + sanity check batch

In [2]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from src.ssl.augmentations import simclr_augment
from src.ssl.dataset_ssl import Kits2DSSLDataset

DATA_ROOT = Path(r"F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

tfm = simclr_augment(size=256)
ssl_ds = Kits2DSSLDataset(DATA_ROOT, "train", transform=tfm)
ssl_loader = DataLoader(ssl_ds, batch_size=32, shuffle=True, num_workers=0,
                        pin_memory=(DEVICE.type=="cuda"))

x1, x2 = next(iter(ssl_loader))
print(x1.shape, x2.shape, x1.min().item(), x1.max().item())

Device: cuda
torch.Size([32, 3, 256, 256]) torch.Size([32, 3, 256, 256]) 0.0 1.0


In [3]:
import importlib
import src.ssl.simclr as simclr
importlib.reload(simclr)

from src.ssl.simclr import SimCLR, ntxent_loss
print("Reloaded ntxent_loss from:", simclr.__file__)

Reloaded ntxent_loss from: f:\projects\hirdl\FedSSL_Paper\src\ssl\simclr.py


## Cell 2: Train SimCLR (1–2 epochs sanity run)

In [4]:
import torch
import torch.optim as optim
from tqdm import tqdm

from src.ssl.simclr import SimCLR, ntxent_loss

# speed tweaks (optional but helpful)
torch.backends.cudnn.benchmark = True

model = SimCLR(proj_dim=128).to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

use_amp = (DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

epochs = 2
temperature = 0.2

for ep in range(1, epochs + 1):
    model.train()
    running = 0.0
    n = 0

    pbar = tqdm(ssl_loader, desc=f"SSL epoch {ep}")
    for x1, x2 in pbar:
        x1 = x1.to(DEVICE, non_blocking=True)
        x2 = x2.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            z1 = model(x1)
            z2 = model(x2)
            loss = ntxent_loss(z1, z2, temperature=temperature)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        bs = x1.size(0)
        running += loss.item() * bs
        n += bs
        pbar.set_postfix(avg_loss=running / max(n, 1))

    print(f"epoch {ep}: ssl_loss={running / max(n, 1):.4f}")

SSL epoch 1: 100%|██████████| 371/371 [01:40<00:00,  3.70it/s, avg_loss=0.967]


epoch 1: ssl_loss=0.9668


SSL epoch 2: 100%|██████████| 371/371 [01:33<00:00,  3.96it/s, avg_loss=0.591]

epoch 2: ssl_loss=0.5913


## Cell 3: Save pretrained encoder weights

In [5]:
from pathlib import Path
import torch

out = Path("outputs/checkpoints/simclr_resnet18_encoder.pt")
out.parent.mkdir(parents=True, exist_ok=True)

torch.save(model.encoder.state_dict(), str(out))
print("Saved:", out)

Saved: outputs\checkpoints\simclr_resnet18_encoder.pt


## Quick SSL sanity check (important)

In [6]:
x1, x2 = next(iter(ssl_loader))
diff = (x1 - x2).abs().mean().item()
print("Mean |x1-x2|:", diff)

Mean |x1-x2|: 0.12691949307918549
